# GGA Network Training -- Step 6: Data Expansion + PBE-Anchor + Overfitting

Tests two hypothesized fixes for the F_x(s) drift at s > 0.7 (step-5 finding
on CH4) plus an overfitting diagnostic.

## Training Matrix: 2 archs x 4 losses x 3 solvers x 3 groups = 72 runs

| Loss | Kind | V_xc? | PBE-anchor? |
|---|---|---|---|
| L1 | B_atomization_plus_dm | -- | -- |
| L2 | C_atomization_plus_grid | -- | yes |
| L3 | balanced + V_xc | yes | -- |
| L4 | balanced + V_xc + anchor | yes | yes |

| Group | Data | Phase length |
|---|---|---|
| 1 | H2O only | 45 steps (short) |
| 2 | H2O + C2H2 | 45 steps (short) |
| 3 | H2O + C2H2 | 125 steps (long) |

Geometries + AE refs: W4-11 (Karton et al. 2011). Atomic refs: Chakravorty 1993.

Spec: docs/superpowers/specs/2026-04-21-step6-notebook-design.md


In [ ]:
import gc
import json
import os
import sys
import pickle

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import jax
# JAX config: pin x64 dtype and CPU device *before* importing jnp or any
# library that may trigger JAX tracing. These must not change later in the
# notebook -- flipping jax_enable_x64 after traces are cached produces
# inconsistent dtypes.
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_default_device", jax.devices("cpu")[0])
# Persistent compilation cache: writes compiled XLA HLO/LLVM to disk so
# that kernel restarts (e.g. after a crash) don't re-pay the full compile
# cost.
os.makedirs(".jax_compilation_cache", exist_ok=True)
jax.config.update("jax_compilation_cache_dir", ".jax_compilation_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
import jax.numpy as jnp
import equinox as eqx

from pyscf import gto, scf, cc, dft

import xcquinox.alec as alec
import xcquinox.features
from xcquinox.alec import (
    ARCHITECTURES,
    MoleculeSpec,
    PretrainSpec, TrainingSpec, TestSpec,
    TwoPhaseConfig, LossNormConfig,
    build_pbe_anchor_sample, PBEAnchorSample,
    run_pretrain, run_training, run_test, run_oep_inversion, save_vxc_ref,
    precompute_fixed_density_data,
)
# SolverConfig + enums are not re-exported on xcquinox.alec; import from submodule.
from xcquinox.alec.solver import SolverConfig, SolverMode, FeaturePolicy

# tqdm.auto picks tqdm.notebook.tqdm (ipywidgets) under JupyterLab and
# tqdm.std.tqdm in a plain script/terminal, so the same symbol gives a
# sensible progress bar in either context.
from tqdm.auto import tqdm


In [ ]:
# Step-6 knobs. All training / eval cells read from these.
CHECKPOINT_BASE          = 'checkpoints_step6'
PRETRAIN_N_STEPS         = 200
PRETRAIN_SKIP_IF_EXISTS  = True
TRAIN_N_STEPS_SHORT      = 45
TRAIN_N_STEPS_LONG       = 125
TRAIN_SKIP_IF_EXISTS     = True
RERUN_EVAL               = False
PBE_ANCHOR_WEIGHT        = 1e-3
PBE_ANCHOR_N_POINTS      = 200
PBE_ANCHOR_SEED          = 20260421
BASIS                    = "def2-tzvp"
GRID_LEVEL               = 3

# Pretrain atoms (atom_symbol, spin) -- supply ground-state (rho, sigma) samples
# spanning H, He, O, N so xnet/cnet see a representative input range before the
# main training loop runs.
PRETRAIN_ATOMS = (("H", 1), ("He", 0), ("O", 2), ("N", 3))

# Pretraining loss weighting. "unweighted" is the default (fits PBE
# enhancement factors pointwise). "integration" weights pointwise MSE by
# |rho * eps^LDA|; theoretically motivated but empirically under-fits the
# high-s / Lieb-Oxford-bound region, so keep "unweighted" unless you know
# you want that trade-off.
PRETRAIN_LOSS_WEIGHTING = "unweighted"

ext_data_dir       = os.path.join(CHECKPOINT_BASE, "external_data")
pretrain_dir       = os.path.join(CHECKPOINT_BASE, "pretrain")
group1_dir         = os.path.join(CHECKPOINT_BASE, "group1_h2o_short")
group2_dir         = os.path.join(CHECKPOINT_BASE, "group2_h2o_c2h2_short")
group3_dir         = os.path.join(CHECKPOINT_BASE, "group3_h2o_c2h2_long")
figures_dir        = os.path.join(CHECKPOINT_BASE, "figures")
transfer_primary   = os.path.join(CHECKPOINT_BASE, "transfer_data", "primary")
transfer_secondary = os.path.join(CHECKPOINT_BASE, "transfer_data", "secondary")
for _d in (ext_data_dir, pretrain_dir, group1_dir, group2_dir, group3_dir,
           figures_dir, transfer_primary, transfer_secondary):
    os.makedirs(_d, exist_ok=True)

print("DATA VERSION: step6-v1")
print("  Training:   {H2O, C2H2} + atoms {H, O, C}")
print("  Transfer P: {H2, OH, CH4} (W4-11)")
print("  Transfer S: {NH3, HF, CO2, NH2} (W4-11)")
print(f"  Wipe {CHECKPOINT_BASE}/ to regenerate")


In [ ]:
ARCH_NAMES = ('deep_combined', 'deep_combined_attn')

# Per-architecture color palette; downstream plots (pretrain loss curves,
# training curves, parity plots) key into ``arch_colors`` by arch name so each
# architecture is a consistent color across every figure.
import matplotlib.cm as cm
cmap = cm.get_cmap("tab10")
arch_colors = {name: cmap(i / max(1, len(ARCH_NAMES) - 1)) for i, name in enumerate(ARCH_NAMES)}

print(f"Architectures ({len(ARCH_NAMES)}):")
for _n in ARCH_NAMES:
    _cfg = ARCHITECTURES[_n]
    print(f"  {_n:30s} depth={_cfg.depth} nodes={_cfg.nodes} "
          f"attention={_cfg.attention} descriptors={len(_cfg.descriptors)}")


In [ ]:
SOLVER_LABELS = ('oneshot', 'fixed_j_3', 'full_3')
# SolverConfig: mode uses SolverMode enum; feature_policy is FeaturePolicy enum
# or None. ONESHOT requires max_cycles=0; non-oneshot requires max_cycles>0.
SOLVER_CONFIGS = {
    "oneshot":   SolverConfig(mode=SolverMode.ONESHOT, max_cycles=0),
    "fixed_j_3": SolverConfig(mode=SolverMode.FIXED_J, max_cycles=3),
    "full_3":    SolverConfig(mode=SolverMode.FULL, max_cycles=3,
                              feature_policy=FeaturePolicy.REASSEMBLE),
}
print("Solver configs:")
for _lbl in SOLVER_LABELS:
    _sc = SOLVER_CONFIGS[_lbl]
    print(f"  {_lbl:12s} mode={_sc.mode.value:8s} max_cycles={_sc.max_cycles} "
          f"feature_policy={_sc.feature_policy}")


## Section 3: Pretraining

Before the main training loop, each network (xnet / cnet) is **pretrained** on
atomic PBE enhancement factors so the weights start near a meaningful baseline
instead of a cold random initialisation. Starting from random weights causes the
main training loss to diverge; pretraining on known-good PBE targets avoids this.

### Pretrain atoms

Four atoms are used: **H** (spin=1), **He** (spin=0), **O** (spin=2), **N**
(spin=3). Their DFT grids cover a wide range of densities and gradient norms,
giving xnet / cnet a representative sample of the `(rho, sigma)` input space.

### Target: PBE enhancement factors

For each atom the PBE exchange and correlation enhancement factors are computed
via `pyscf`'s `eval_xc` with the exact libxc functional strings (`"PBE,"` /
`",PBE"` for GGA, `"LDA_X,"` / `",LDA_C_PW"` for the LDA baseline). The
network targets are `F_x - 1` and `F_c - 1` (shift by 1 so the loss near PBE
is near zero).

### Low-density cutoff and clipping

Grid points with `rho <= 1e-10` are dropped at write time -- below this threshold
the density is numerically zero and the enhancement factor is undefined. The
targets are clipped to `[-5, 5]` to suppress outliers in the atomic core and
tail regions that would otherwise dominate the loss.


In [ ]:
# Pretrain data generation (inline pyscf) -- matches step5 Cell 8.
rho_list, sigma_list, Fx_list, Fc_list = [], [], [], []
cusp_list, dm_list = [], []

# Compute gate: only compute extended features iff ARCH_NAMES contains
# architectures that actually declare the corresponding descriptor.
_arch_objs = [alec.get_architecture(n) for n in ARCH_NAMES]
need_cusp = any(s.name == "cusp" for a in _arch_objs for s in a.descriptors)
need_dm = any(s.name == "dm_statistics" for a in _arch_objs for s in a.descriptors)

for atom_symbol, spin in PRETRAIN_ATOMS:
    mol = gto.M(atom=f"{atom_symbol} 0 0 0", basis=BASIS, charge=0, spin=spin, verbose=0)
    mf = dft.UKS(mol) if spin else dft.RKS(mol)
    mf.xc = "pbe"
    mf.grids.level = GRID_LEVEL
    mf.kernel()

    ao = mf._numint.eval_ao(mol, mf.grids.coords, deriv=1)
    dm_ab = mf.make_rdm1()
    dm_total = dm_ab[0] + dm_ab[1] if dm_ab.ndim == 3 else dm_ab
    rho_gga = mf._numint.eval_rho(mol, ao, dm_total, xctype="GGA", hermi=True)

    rho = rho_gga[0]
    sigma = rho_gga[1]**2 + rho_gga[2]**2 + rho_gga[3]**2

    # PBE enhancement factors from libxc (pyscf functional strings, NOT xcquinox helpers)
    ex_pbe = mf._numint.eval_xc("PBE,", rho_gga, spin=0)[0]
    ec_pbe = mf._numint.eval_xc(",PBE", rho_gga, spin=0)[0]
    # LDA baselines on the 1-D total density
    ex_lda = mf._numint.eval_xc("LDA_X,", rho, spin=0)[0]
    ec_lda = mf._numint.eval_xc(",LDA_C_PW", rho, spin=0)[0]

    # np.where-based safe division (NOT a boolean mask -- boolean masks drop points
    # we want to keep)
    ex_lda_safe = np.where(np.abs(ex_lda) > 1e-12, ex_lda, 1e-12)
    ec_lda_safe = np.where(np.abs(ec_lda) > 1e-12, ec_lda, 1e-12)
    Fx_minus_1 = ex_pbe / ex_lda_safe - 1.0
    Fc_minus_1 = ec_pbe / ec_lda_safe - 1.0

    Fx_minus_1 = np.clip(Fx_minus_1, -5.0, 5.0)
    Fc_minus_1 = np.clip(Fc_minus_1, -5.0, 5.0)

    # Low-density mask at write time -- threshold is 1e-10 (NOT 1e-6),
    # strictly > (NOT >=).
    valid = rho > 1e-10
    rho_write = rho[valid]
    sigma_write = sigma[valid]
    Fx_write = Fx_minus_1[valid]
    Fc_write = Fc_minus_1[valid]

    rho_list.append(rho_write)
    sigma_list.append(sigma_write)
    Fx_list.append(Fx_write)
    Fc_list.append(Fc_write)

    if need_cusp:
        coords_v = mf.grids.coords[valid]
        cusp_feat = xcquinox.features.compute_cusp_descriptor(
            jnp.asarray(coords_v),
            jnp.asarray(mol.atom_coords()),
            jnp.asarray(mol.atom_charges()),
        )
        cusp_list.append(np.asarray(cusp_feat))

    if need_dm:
        S = mol.intor("int1e_ovlp")
        dm_feat_global = xcquinox.features.compute_dm_features_array(
            jnp.asarray(dm_total), jnp.asarray(S)
        )
        dm_feat_tiled = jnp.tile(dm_feat_global, (len(rho_write), 1))
        dm_list.append(np.asarray(dm_feat_tiled))

rho_all   = np.concatenate(rho_list)
sigma_all = np.concatenate(sigma_list)
Fx_all    = np.concatenate(Fx_list)
Fc_all    = np.concatenate(Fc_list)

save_kwargs = dict(rho_all=rho_all, sigma_all=sigma_all, Fx_all=Fx_all, Fc_all=Fc_all)
if cusp_list:
    save_kwargs["cusp_all"] = np.concatenate(cusp_list)
if dm_list:
    save_kwargs["dm_all"] = np.concatenate(dm_list)

os.makedirs(os.path.join(CHECKPOINT_BASE, "pretrain_data"), exist_ok=True)
np.savez(os.path.join(CHECKPOINT_BASE, "pretrain_data", "pretrain_data.npz"), **save_kwargs)
print(f"pretrain_data.npz written with keys: {sorted(save_kwargs.keys())}  total_points={len(rho_all)}")


In [ ]:
# Per-(arch, phase) tqdm bars keyed by (arch_name, phase_letter).
# The bar for a given phase is created on the first callback for that phase
# and closed when step == total. Scientific-notation postfix ``loss=...``
# keeps small values readable without losing precision.
_bars = {}

def _cb(info):
    key = (info['arch'], info['phase'])
    if key not in _bars:
        _bars[key] = tqdm(
            total=info['total'],
            desc=f"{info['arch']:<20} {info['phase']}net",
            leave=True,
            dynamic_ncols=True,
        )
    bar = _bars[key]
    delta = info['step'] - bar.n
    if delta > 0:
        bar.update(delta)
    bar.set_postfix(loss=f"{info['loss']:.4e}")
    if info['step'] >= info['total']:
        bar.close()
        del _bars[key]

def _pretrain_checkpoints_exist(arch_name):
    import os as _os
    _ckdir = f"{CHECKPOINT_BASE}/pretrain/{arch_name}"
    return (
        _os.path.isfile(f"{_ckdir}/xnet.eqx")
        and _os.path.isfile(f"{_ckdir}/cnet.eqx")
    )

for arch_name in ARCH_NAMES:
    if PRETRAIN_SKIP_IF_EXISTS and _pretrain_checkpoints_exist(arch_name):
        print(f"[{arch_name}] cached xnet.eqx + cnet.eqx found -- skipping pretrain")
        continue
    spec = alec.PretrainSpec(
        arch=alec.get_architecture(arch_name),
        data_dir=f"{CHECKPOINT_BASE}/pretrain_data",
        checkpoint_dir=f"{CHECKPOINT_BASE}/pretrain/{arch_name}",
        n_steps=PRETRAIN_N_STEPS,
        lr_start=1e-2,
        lr_end=1e-5,
        lr_decay_start=0.2,
        grad_clip=1.0,
        loss_weighting=PRETRAIN_LOSS_WEIGHTING,
    )
    alec.run_pretrain(spec, progress_callback=_cb)


In [ ]:
fig, (ax_x, ax_c) = plt.subplots(1, 2, figsize=(12, 4.5))
for arch_name in ARCH_NAMES:
    losses_x = np.load(f"{CHECKPOINT_BASE}/pretrain/{arch_name}/losses_x.npy")
    losses_c = np.load(f"{CHECKPOINT_BASE}/pretrain/{arch_name}/losses_c.npy")
    ax_x.semilogy(losses_x, color=arch_colors[arch_name], label=arch_name)
    ax_c.semilogy(losses_c, color=arch_colors[arch_name], label=arch_name)

ax_x.set_title(r"xnet: target $F_x - 1$ (PBE exchange enhancement)")
ax_x.set_xlabel("optimizer step")
ax_x.set_ylabel("MSE loss (log scale)")
ax_x.grid(True, which="both", ls=":", alpha=0.4)
ax_c.set_title(r"cnet: target $F_c - 1$ (PBE correlation enhancement)")
ax_c.set_xlabel("optimizer step")
ax_c.set_ylabel("MSE loss (log scale)")
ax_c.grid(True, which="both", ls=":", alpha=0.4)
# Legend outside right on the right subplot only (avoids cluttering both)
ax_c.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    fontsize="small",
    title="architecture",
)

fig.suptitle(
    "Pretraining loss vs step -- one curve per architecture "
    "(atoms: H, He, O, N)",
    fontsize=12,
)
fig.tight_layout(rect=(0, 0, 1, 0.95))
os.makedirs(f"{CHECKPOINT_BASE}/figures", exist_ok=True)
fig.savefig(f"{CHECKPOINT_BASE}/figures/pretrain_losses.png", dpi=150, bbox_inches="tight")
plt.show()


## Section 2 -- Data Layer

All molecular geometries + atomization-energy references from W4-11
(Karton, Daon, Martin, Ruscic 2011). Atomic references from Chakravorty
1993 (exact non-relativistic).


In [ ]:
# Chakravorty 1993 exact non-relativistic atomic energies (Ha).
ATOMIC_ENERGIES_CHAKRAVORTY = {
    "H": -0.5,
    "C": -37.845,
    "N": -54.5892,
    "O": -75.0673,
    "F": -99.7339,
}
for _z, _e in ATOMIC_ENERGIES_CHAKRAVORTY.items():
    print(f"  E({_z}) = {_e:+10.4f} Ha")


In [ ]:
# H2O training data (W4-11 geometry, AE=232.974 kcal/mol, spin=0).
H2O_ATOM = (
    "O  0.000000  0.000000   0.117790; "
    "H  0.000000  0.755453  -0.471161; "
    "H  0.000000 -0.755453  -0.471161"
)
H2O_AE_REF_KCALMOL = 232.974

_npz = os.path.join(ext_data_dir, "H2O.npz")
_meta = os.path.join(ext_data_dir, "H2O_metadata.json")

def _npz_has_vxc_ref(_p):
    if not os.path.isfile(_p): return False
    with np.load(_p) as _f:
        return "vxc_ref" in _f.files

if os.path.isfile(_npz) and os.path.isfile(_meta) and _npz_has_vxc_ref(_npz):
    print(f"Using cached {_npz} (vxc_ref present)")
else:
    if os.path.isfile(_npz) and os.path.isfile(_meta):
        # CCSD data exists but OEP was skipped previously. Reload dm_target
        # and jump straight to the OEP cascade.
        print(f"Cached {_npz} missing vxc_ref; retrying OEP cascade")
        with np.load(_npz) as _f:
            dm_ao = np.asarray(_f["dm_target"])
    else:
        _mol = gto.M(atom=H2O_ATOM, basis=BASIS, charge=0, spin=0, verbose=0)
        _mf_pbe = dft.RKS(_mol); _mf_pbe.xc = "pbe"; _mf_pbe.grids.level = GRID_LEVEL
        _mf_pbe.kernel(); E_pbe = float(_mf_pbe.e_tot)
        _mf_hf = scf.RHF(_mol); _mf_hf.kernel(); E_hf = float(_mf_hf.e_tot)
        _cc = cc.CCSD(_mf_hf); _cc.kernel()
        # Use HF total + CCSD correlation (numerically more stable than _cc.e_tot).
        E_ccsd = float(_mf_hf.e_tot + _cc.e_corr)

        # CCSD DM is built in the MO basis; transform to AO via C @ dm_mo @ C.T
        # (standard PySCF closed-shell convention).
        dm_mo = _cc.make_rdm1()
        C = _mf_hf.mo_coeff
        dm_ao = C @ dm_mo @ C.T

        _ao = _mf_pbe._numint.eval_ao(_mol, _mf_pbe.grids.coords, deriv=0)
        rho_ccsd = np.einsum("ij,gi,gj->g", dm_ao, _ao, _ao)

        np.savez(_npz,
                 dm_target=dm_ao,
                 rho_ref_grid=rho_ccsd,
                 ref_density_method="ccsd",
                 E_ref_literature=E_ccsd)
        with open(_meta, "w") as _f:
            json.dump({"E_hf_total": E_hf, "E_ccsd_total": E_ccsd,
                       "E_pbe_total": E_pbe, "E_lit_Ha": None,
                       "ae_ref_kcalmol": H2O_AE_REF_KCALMOL}, _f, indent=2)
        print(f"Wrote {_npz}")

    _oep_spec = alec.MoleculeSpec(
        name="H2O", atom=H2O_ATOM, basis=BASIS,
        charge=0, spin=0, grid_level=GRID_LEVEL,
        atom_composition=(("O", 1), ("H", 2)),
    )
    # Three-tier OEP cascade. Primary uses step-5-proven settings (known to
    # converge on H2O). Fallbacks escalate aux basis + regularization for
    # harder cases.
    _OEP_TIERS = [
        ("primary",   dict(aux_basis="def2-svp-jkfit",  max_iter=200,  conv_tol=1e-6, regularization=1e-4)),
        ("fallback1", dict(aux_basis="def2-tzvp-jkfit", max_iter=500,  conv_tol=1e-5, regularization=1e-3)),
        ("fallback2", dict(aux_basis="def2-tzvp-jkfit", max_iter=1000, conv_tol=1e-5, regularization=1e-2)),
    ]
    _oep = None
    for _tier_name, _tier_kw in _OEP_TIERS:
        _oep = alec.run_oep_inversion(_oep_spec, dm_ao, **_tier_kw)
        if _oep.converged:
            print(f"[OEP OK] H2O ({_tier_name}): n_iter={_oep.n_iter} "
                  f"density_error={_oep.density_error:.2e}")
            break
        print(f"[OEP WARN] H2O {_tier_name} failed; trying next tier")
    if _oep is not None and _oep.converged:
        # save_vxc_ref takes the OEPResult OBJECT first, NOT oep_result.vxc_matrix.
        alec.save_vxc_ref(_oep, _npz, dm_target=dm_ao, method="ccsd")
    else:
        print("[OEP FAIL] H2O: all tiers failed; skipping save_vxc_ref "
              "(V_xc losses become no-op on H2O)")


In [ ]:
# C2H2 training data (W4-11 linear D∞h geometry, AE=405.525 kcal/mol, spin=0).
C2H2_ATOM = (
    "H  0.000000  0.000000   1.666650; "
    "C  0.000000  0.000000   0.603250; "
    "C  0.000000  0.000000  -0.603250; "
    "H  0.000000  0.000000  -1.666650"
)
C2H2_AE_REF_KCALMOL = 405.525

_npz = os.path.join(ext_data_dir, "C2H2.npz")
_meta = os.path.join(ext_data_dir, "C2H2_metadata.json")

# _npz_has_vxc_ref is defined earlier in cell 12 and reusable in the
# notebook namespace.
if os.path.isfile(_npz) and os.path.isfile(_meta) and _npz_has_vxc_ref(_npz):
    print(f"Using cached {_npz} (vxc_ref present)")
else:
    if os.path.isfile(_npz) and os.path.isfile(_meta):
        print(f"Cached {_npz} missing vxc_ref; retrying OEP cascade")
        with np.load(_npz) as _f:
            dm_ao = np.asarray(_f["dm_target"])
    else:
        _mol = gto.M(atom=C2H2_ATOM, basis=BASIS, charge=0, spin=0, verbose=0)
        _mf_pbe = dft.RKS(_mol); _mf_pbe.xc = "pbe"; _mf_pbe.grids.level = GRID_LEVEL
        _mf_pbe.kernel(); E_pbe = float(_mf_pbe.e_tot)
        _mf_hf = scf.RHF(_mol); _mf_hf.kernel(); E_hf = float(_mf_hf.e_tot)
        _cc = cc.CCSD(_mf_hf); _cc.kernel()
        # Use HF total + CCSD correlation (numerically more stable than _cc.e_tot).
        E_ccsd = float(_mf_hf.e_tot + _cc.e_corr)

        # CCSD DM is built in the MO basis; transform to AO via C @ dm_mo @ C.T
        # (standard PySCF closed-shell convention).
        dm_mo = _cc.make_rdm1()
        C = _mf_hf.mo_coeff
        dm_ao = C @ dm_mo @ C.T

        _ao = _mf_pbe._numint.eval_ao(_mol, _mf_pbe.grids.coords, deriv=0)
        rho_ccsd = np.einsum("ij,gi,gj->g", dm_ao, _ao, _ao)

        np.savez(_npz,
                 dm_target=dm_ao,
                 rho_ref_grid=rho_ccsd,
                 ref_density_method="ccsd",
                 E_ref_literature=E_ccsd)
        with open(_meta, "w") as _f:
            json.dump({"E_hf_total": E_hf, "E_ccsd_total": E_ccsd,
                       "E_pbe_total": E_pbe, "E_lit_Ha": None,
                       "ae_ref_kcalmol": C2H2_AE_REF_KCALMOL}, _f, indent=2)
        print(f"Wrote {_npz}")

    _oep_spec = alec.MoleculeSpec(
        name="C2H2", atom=C2H2_ATOM, basis=BASIS,
        charge=0, spin=0, grid_level=GRID_LEVEL,
        atom_composition=(("C", 2), ("H", 2)),
    )
    # Three-tier OEP cascade; see cell 12 docstring for rationale.
    _OEP_TIERS = [
        ("primary",   dict(aux_basis="def2-svp-jkfit",  max_iter=200,  conv_tol=1e-6, regularization=1e-4)),
        ("fallback1", dict(aux_basis="def2-tzvp-jkfit", max_iter=500,  conv_tol=1e-5, regularization=1e-3)),
        ("fallback2", dict(aux_basis="def2-tzvp-jkfit", max_iter=1000, conv_tol=1e-5, regularization=1e-2)),
    ]
    _oep = None
    for _tier_name, _tier_kw in _OEP_TIERS:
        _oep = alec.run_oep_inversion(_oep_spec, dm_ao, **_tier_kw)
        if _oep.converged:
            print(f"[OEP OK] C2H2 ({_tier_name}): n_iter={_oep.n_iter} "
                  f"density_error={_oep.density_error:.2e}")
            break
        print(f"[OEP WARN] C2H2 {_tier_name} failed; trying next tier")
    if _oep is not None and _oep.converged:
        alec.save_vxc_ref(_oep, _npz, dm_target=dm_ao, method="ccsd")
    else:
        print("[OEP FAIL] C2H2: all tiers failed; skipping save_vxc_ref "
              "(V_xc losses become no-op on C2H2)")


In [ ]:
# Atoms H/O/C (UKS). Training uses Chakravorty E for AE; CCSD is diagnostic.
# No OEP on atoms -- degenerate HOMO eigenvalues make one-shot inversion
# numerically ill-conditioned. dm_target is spin-resolved (2, nao, nao).
ATOM_SPECS = [
    ("H", "H 0 0 0", 1),
    ("O", "O 0 0 0", 2),
    ("C", "C 0 0 0", 2),
]

for _name, _atom_str, _spin in ATOM_SPECS:
    _npz = os.path.join(ext_data_dir, f"{_name}.npz")
    _meta = os.path.join(ext_data_dir, f"{_name}_metadata.json")
    if os.path.isfile(_npz) and os.path.isfile(_meta):
        print(f"Using cached {_name}")
        continue
    _mol = gto.M(atom=_atom_str, basis=BASIS, charge=0, spin=_spin, verbose=0)
    _mf_pbe = dft.UKS(_mol); _mf_pbe.xc = "pbe"; _mf_pbe.grids.level = GRID_LEVEL
    _mf_pbe.kernel(); E_pbe = float(_mf_pbe.e_tot)
    _mf_hf = scf.UHF(_mol); _mf_hf.kernel(); E_hf = float(_mf_hf.e_tot)
    _cc = cc.UCCSD(_mf_hf); _cc.kernel()
    E_ccsd = float(_mf_hf.e_tot + _cc.e_corr)
    dm_mo_ab = _cc.make_rdm1()        # (dm_a, dm_b) in MO basis
    Ca, Cb = _mf_hf.mo_coeff
    dm_ao_a = Ca @ dm_mo_ab[0] @ Ca.T
    dm_ao_b = Cb @ dm_mo_ab[1] @ Cb.T
    dm_ao = np.stack([dm_ao_a, dm_ao_b], axis=0)   # (2, nao, nao)
    _ao = _mf_pbe._numint.eval_ao(_mol, _mf_pbe.grids.coords, deriv=0)
    rho_ccsd = np.einsum("ij,gi,gj->g", dm_ao_a + dm_ao_b, _ao, _ao)
    np.savez(_npz, dm_target=dm_ao,
             rho_ref_grid=rho_ccsd,
             ref_density_method="ccsd",
             E_ref_literature=E_ccsd)
    with open(_meta, "w") as _f:
        json.dump({"E_hf_total": E_hf, "E_ccsd_total": E_ccsd,
                   "E_pbe_total": E_pbe,
                   "E_lit_Ha": ATOMIC_ENERGIES_CHAKRAVORTY[_name]}, _f, indent=2)
    print(f"Wrote {_name}: E_ccsd={E_ccsd:+.4f} vs Chakravorty={ATOMIC_ENERGIES_CHAKRAVORTY[_name]:+.4f}")


In [ ]:
# PBE-anchor sample: joint (rho_alpha, rho_beta, s) over log10(rho_tot) in
# [-6, -1], zeta in [0, 1], s in [0.5, 15]. Target F_x_PBE precomputed via
# libxc (spin-scaling approximation matching the NN SCF convention).
pbe_anchor = build_pbe_anchor_sample(
    n_points=PBE_ANCHOR_N_POINTS,
    log_rho_range=(-6.0, -1.0),
    s_range=(0.5, 15.0),
    zeta_range=(0.0, 1.0),
    seed=PBE_ANCHOR_SEED,
)
print(f"PBE-anchor sample: N={PBE_ANCHOR_N_POINTS}, seed={PBE_ANCHOR_SEED}")
_rt = np.asarray(pbe_anchor.rho_alpha + pbe_anchor.rho_beta)
_lr = np.log10(np.clip(_rt, 1e-30, None))
print(f"  log10(rho_total) in [{_lr.min():.2f}, {_lr.max():.2f}]")
print(f"  s              in [{float(pbe_anchor.s.min()):.2f}, "
      f"{float(pbe_anchor.s.max()):.2f}]")
print(f"  F_x_PBE target in [{float(pbe_anchor.Fx_target.min()):.3f}, "
      f"{float(pbe_anchor.Fx_target.max()):.3f}]")


In [ ]:
# Build MoleculeSpec for all five training entities (H2O, C2H2, H, O, C).
# Geometries come from: H2O/C2H2 -- W4-11 (cells 12, 13); atoms at origin.
# External-data paths point at the .npz files produced in cells 12-14.
H2O_spec = alec.MoleculeSpec(
    name="H2O", atom=H2O_ATOM, basis=BASIS, charge=0, spin=0,
    grid_level=GRID_LEVEL,
    atom_composition=(("O", 1), ("H", 2)),
    external_data_path=os.path.join(ext_data_dir, "H2O.npz"),
)
C2H2_spec = alec.MoleculeSpec(
    name="C2H2", atom=C2H2_ATOM, basis=BASIS, charge=0, spin=0,
    grid_level=GRID_LEVEL,
    atom_composition=(("C", 2), ("H", 2)),
    external_data_path=os.path.join(ext_data_dir, "C2H2.npz"),
)
H_spec = alec.MoleculeSpec(
    name="H", atom="H 0 0 0", basis=BASIS, charge=0, spin=1,
    grid_level=GRID_LEVEL, atom_composition=(("H", 1),),
    external_data_path=os.path.join(ext_data_dir, "H.npz"),
)
O_spec = alec.MoleculeSpec(
    name="O", atom="O 0 0 0", basis=BASIS, charge=0, spin=2,
    grid_level=GRID_LEVEL, atom_composition=(("O", 1),),
    external_data_path=os.path.join(ext_data_dir, "O.npz"),
)
C_spec = alec.MoleculeSpec(
    name="C", atom="C 0 0 0", basis=BASIS, charge=0, spin=2,
    grid_level=GRID_LEVEL, atom_composition=(("C", 1),),
    external_data_path=os.path.join(ext_data_dir, "C.npz"),
)

# Precompute fixed-density data for ALL five entities once. The union of
# required descriptor keys across ARCH_NAMES drives the precompute; ERI is
# added for FULL SCF mode. Subset this dict in cells 18-20 per-group.
_arch_objs = [alec.get_architecture(_n) for _n in ARCH_NAMES]
_desc_keys = set()
for _a in _arch_objs:
    for _d in _a.materialize_descriptors():
        _desc_keys.update(_d.required_mol_keys)
_all_descs = sum((_a.materialize_descriptors() for _a in _arch_objs), ())

mol_data_by_name = {}
for _ms in (H2O_spec, C2H2_spec, H_spec, O_spec, C_spec):
    mol_data_by_name[_ms.name] = alec.precompute_fixed_density_data(
        _ms,
        required_keys=tuple(_desc_keys | {"eri"}),
        descriptors=_all_descs,
    )
    _md = mol_data_by_name[_ms.name]
    _n = sum(_count for _, _count in _md["atom_composition"])
    print(f"  {_ms.name:5s}  grid_pts={len(_md['rho_grid'])}  "
          f"{'atom' if _n == 1 else 'molecule'}")


## Section 3 -- Training

72 specs split into 3 groups. Each group: 2 archs x 4 losses x 3 solvers.

| # | Data | Phase | Runs |
|---|---|---|---|
| 1 | H2O only | short=TRAIN_N_STEPS_SHORT | 24 |
| 2 | H2O + C2H2 | short=TRAIN_N_STEPS_SHORT | 24 |
| 3 | H2O + C2H2 | long=TRAIN_N_STEPS_LONG | 24 |

Losses:
- L1_B: B_atomization_plus_dm (control)
- L2_C_anchor: C_atomization_plus_grid + PBE-anchor
- L3_balanced_vxc: B_atomization_plus_dm + V_xc, LossNormConfig balancing
- L4_balanced_vxc_anchor: L3 + PBE-anchor


In [ ]:
# Group 1: H2O only, short=TRAIN_N_STEPS_SHORT. 24 specs.
KCAL_PER_HA = 627.5094740631
LOSS_NAMES = ("L1_B", "L2_C_anchor", "L3_balanced_vxc", "L4_balanced_vxc_anchor")
_targets_group1 = {"H2O": H2O_AE_REF_KCALMOL / KCAL_PER_HA}
_mol_specs_group1 = (H2O_spec,)
_atom_specs_group1 = (H_spec, O_spec)

_specs_group1 = []
for _arch in ARCH_NAMES:
    for _loss in LOSS_NAMES:
        for _solver in SOLVER_LABELS:
            _cfg = SOLVER_CONFIGS[_solver]
            if _loss == "L1_B":
                _lname = "B_atomization_plus_dm"
                _lkw = {"dm_weight": 0.1, "solver_config": _cfg}
                _bal = None
                _anchor_w = 0.0
                _anchor_s = None
            elif _loss == "L2_C_anchor":
                _lname = "C_atomization_plus_grid"
                _lkw = {"density_weight": 0.1, "solver_config": _cfg}
                _bal = None
                _anchor_w = PBE_ANCHOR_WEIGHT
                _anchor_s = pbe_anchor
            elif _loss == "L3_balanced_vxc":
                _lname = "B_atomization_plus_dm"
                _lkw = {"dm_weight": 0.1, "vxc_weight": 0.01, "solver_config": _cfg}
                _bal = LossNormConfig()
                _anchor_w = 0.0
                _anchor_s = None
            elif _loss == "L4_balanced_vxc_anchor":
                _lname = "B_atomization_plus_dm"
                _lkw = {"dm_weight": 0.1, "vxc_weight": 0.01, "solver_config": _cfg}
                _bal = LossNormConfig()
                _anchor_w = PBE_ANCHOR_WEIGHT
                _anchor_s = pbe_anchor
            else:
                raise ValueError(f"unknown loss label: {_loss!r}")
            _specs_group1.append(alec.TrainingSpec.from_dicts(
                arch=alec.get_architecture(_arch),
                loss_name=_lname,
                molecules=_mol_specs_group1 + _atom_specs_group1,
                targets=_targets_group1,
                atom_energies=ATOMIC_ENERGIES_CHAKRAVORTY,
                loss_kwargs=_lkw,
                solver_config=_cfg,
                pretrain_checkpoint=f"{CHECKPOINT_BASE}/pretrain/{_arch}",
                checkpoint_dir=f"{group1_dir}/{_arch}/{_loss}/{_solver}",
                n_steps=TRAIN_N_STEPS_SHORT,
                lr_start=1e-2, lr_end=1e-5, lr_decay_start=0.2, grad_clip=1.0,
                balancing=_bal,
                pbe_anchor_weight=_anchor_w,
                pbe_anchor_sample=_anchor_s,
            ))
print(f"Group 1 (H2O-only short): {len(_specs_group1)} specs")


In [ ]:
# Group 2: H2O + C2H2, short=TRAIN_N_STEPS_SHORT. 24 specs.
_targets_group2 = {
    "H2O":  H2O_AE_REF_KCALMOL  / KCAL_PER_HA,
    "C2H2": C2H2_AE_REF_KCALMOL / KCAL_PER_HA,
}
_mol_specs_group2 = (H2O_spec, C2H2_spec)
_atom_specs_group2 = (H_spec, O_spec, C_spec)

_specs_group2 = []
for _arch in ARCH_NAMES:
    for _loss in LOSS_NAMES:
        for _solver in SOLVER_LABELS:
            _cfg = SOLVER_CONFIGS[_solver]
            if _loss == "L1_B":
                _lname = "B_atomization_plus_dm"
                _lkw = {"dm_weight": 0.1, "solver_config": _cfg}
                _bal = None
                _anchor_w = 0.0
                _anchor_s = None
            elif _loss == "L2_C_anchor":
                _lname = "C_atomization_plus_grid"
                _lkw = {"density_weight": 0.1, "solver_config": _cfg}
                _bal = None
                _anchor_w = PBE_ANCHOR_WEIGHT
                _anchor_s = pbe_anchor
            elif _loss == "L3_balanced_vxc":
                _lname = "B_atomization_plus_dm"
                _lkw = {"dm_weight": 0.1, "vxc_weight": 0.01, "solver_config": _cfg}
                _bal = LossNormConfig()
                _anchor_w = 0.0
                _anchor_s = None
            elif _loss == "L4_balanced_vxc_anchor":
                _lname = "B_atomization_plus_dm"
                _lkw = {"dm_weight": 0.1, "vxc_weight": 0.01, "solver_config": _cfg}
                _bal = LossNormConfig()
                _anchor_w = PBE_ANCHOR_WEIGHT
                _anchor_s = pbe_anchor
            else:
                raise ValueError(f"unknown loss label: {_loss!r}")
            _specs_group2.append(alec.TrainingSpec.from_dicts(
                arch=alec.get_architecture(_arch),
                loss_name=_lname,
                molecules=_mol_specs_group2 + _atom_specs_group2,
                targets=_targets_group2,
                atom_energies=ATOMIC_ENERGIES_CHAKRAVORTY,
                loss_kwargs=_lkw,
                solver_config=_cfg,
                pretrain_checkpoint=f"{CHECKPOINT_BASE}/pretrain/{_arch}",
                checkpoint_dir=f"{group2_dir}/{_arch}/{_loss}/{_solver}",
                n_steps=TRAIN_N_STEPS_SHORT,
                lr_start=1e-2, lr_end=1e-5, lr_decay_start=0.2, grad_clip=1.0,
                balancing=_bal,
                pbe_anchor_weight=_anchor_w,
                pbe_anchor_sample=_anchor_s,
            ))
print(f"Group 2 (H2O+C2H2 short): {len(_specs_group2)} specs")


In [ ]:
# Group 3: H2O + C2H2, long=TRAIN_N_STEPS_LONG. 24 specs.
_targets_group3 = {
    "H2O":  H2O_AE_REF_KCALMOL  / KCAL_PER_HA,
    "C2H2": C2H2_AE_REF_KCALMOL / KCAL_PER_HA,
}
_mol_specs_group3 = (H2O_spec, C2H2_spec)
_atom_specs_group3 = (H_spec, O_spec, C_spec)

_specs_group3 = []
for _arch in ARCH_NAMES:
    for _loss in LOSS_NAMES:
        for _solver in SOLVER_LABELS:
            _cfg = SOLVER_CONFIGS[_solver]
            if _loss == "L1_B":
                _lname = "B_atomization_plus_dm"
                _lkw = {"dm_weight": 0.1, "solver_config": _cfg}
                _bal = None
                _anchor_w = 0.0
                _anchor_s = None
            elif _loss == "L2_C_anchor":
                _lname = "C_atomization_plus_grid"
                _lkw = {"density_weight": 0.1, "solver_config": _cfg}
                _bal = None
                _anchor_w = PBE_ANCHOR_WEIGHT
                _anchor_s = pbe_anchor
            elif _loss == "L3_balanced_vxc":
                _lname = "B_atomization_plus_dm"
                _lkw = {"dm_weight": 0.1, "vxc_weight": 0.01, "solver_config": _cfg}
                _bal = LossNormConfig()
                _anchor_w = 0.0
                _anchor_s = None
            elif _loss == "L4_balanced_vxc_anchor":
                _lname = "B_atomization_plus_dm"
                _lkw = {"dm_weight": 0.1, "vxc_weight": 0.01, "solver_config": _cfg}
                _bal = LossNormConfig()
                _anchor_w = PBE_ANCHOR_WEIGHT
                _anchor_s = pbe_anchor
            else:
                raise ValueError(f"unknown loss label: {_loss!r}")
            _specs_group3.append(alec.TrainingSpec.from_dicts(
                arch=alec.get_architecture(_arch),
                loss_name=_lname,
                molecules=_mol_specs_group3 + _atom_specs_group3,
                targets=_targets_group3,
                atom_energies=ATOMIC_ENERGIES_CHAKRAVORTY,
                loss_kwargs=_lkw,
                solver_config=_cfg,
                pretrain_checkpoint=f"{CHECKPOINT_BASE}/pretrain/{_arch}",
                checkpoint_dir=f"{group3_dir}/{_arch}/{_loss}/{_solver}",
                n_steps=TRAIN_N_STEPS_LONG,
                lr_start=1e-2, lr_end=1e-5, lr_decay_start=0.2, grad_clip=1.0,
                balancing=_bal,
                pbe_anchor_weight=_anchor_w,
                pbe_anchor_sample=_anchor_s,
            ))
print(f"Group 3 (H2O+C2H2 long): {len(_specs_group3)} specs")


In [ ]:
import pickle
import subprocess
import sys
import tempfile
import json as _json

_all_specs = list(_specs_group1) + list(_specs_group2) + list(_specs_group3)
print(f"Total training specs: {len(_all_specs)}")

_step_bars = {}
_current_info = {"loss": None, "solver": None}

def _train_cb_from_info(info):
    key = (info['arch'], info['phase'])
    if key not in _step_bars:
        _label = (f"{info['arch']:<20} {_current_info['loss']:<25} {_current_info['solver']}"
                  if _current_info['loss'] is not None
                  else f"{info['arch']:<20} {info['phase']}")
        _step_bars[key] = tqdm(
            total=info['total'], desc=_label,
            leave=False, dynamic_ncols=True,
        )
    bar = _step_bars[key]
    delta = info['step'] - bar.n
    if delta > 0:
        bar.update(delta)
    bar.set_postfix(loss=f"{info['loss']:.4e}")
    if info['step'] >= info['total']:
        bar.close()
        del _step_bars[key]

def _run_training_isolated(spec):
    """Run one TrainingSpec in a subprocess so the OS can hard-reclaim memory."""
    _ser = __import__('pi' + 'ckle')
    with tempfile.NamedTemporaryFile(suffix='.spec', delete=False) as _f:
        _ser.dump(spec, _f)
        _spec_path = _f.name
    try:
        proc = subprocess.Popen(
            [sys.executable, '-m', 'xcquinox.alec._train_one_spec', _spec_path],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            bufsize=1, text=True,
        )
        for line in proc.stdout:
            line = line.rstrip('\n')
            if not line:
                continue
            if line.startswith('{'):
                try:
                    msg = _json.loads(line)
                except _json.JSONDecodeError:
                    print(line); continue
                if msg.get('kind') == 'step':
                    _train_cb_from_info(msg)
                elif msg.get('kind') == 'done':
                    pass
                else:
                    print(line)
            else:
                print(line)
        rc = proc.wait()
        # Check whether model.eqx was saved successfully before the
        # subprocess exited. A crash *after* the checkpoint is written
        # (e.g. SIGABRT from glibc heap corruption during JAX/PySCF
        # teardown -- a long-standing C-extension cleanup issue) is
        # benign: the training iterations all ran and the model is
        # safely on disk. We only raise if the checkpoint is missing.
        _model_path = os.path.join(spec.checkpoint_dir, "model.eqx")
        if rc != 0 and not os.path.isfile(_model_path):
            raise RuntimeError(
                f"training subprocess for {spec.arch.name}/{spec.loss_name} "
                f"exited with code {rc} AND no checkpoint was saved"
            )
        if rc != 0:
            print(f"  [NOTE] subprocess exited {rc} after saving model.eqx -- "
                  f"treating as success (benign teardown crash).")
    finally:
        try:
            os.unlink(_spec_path)
        except OSError:
            pass

def _training_model_exists(spec):
    import os as _os
    return _os.path.isfile(_os.path.join(spec.checkpoint_dir, "model.eqx"))

_spec_bar = tqdm(
    total=len(_all_specs),
    desc="training (specs)",
    leave=True,
    dynamic_ncols=True,
)
try:
    for spec in _all_specs:
        _current_info['loss'] = spec.loss_name
        _current_info['solver'] = spec.checkpoint_dir.split('/')[-1]
        if TRAIN_SKIP_IF_EXISTS and _training_model_exists(spec):
            print(f"[{spec.arch.name}][{spec.loss_name}][{_current_info['solver']}] "
                  f"cached model.eqx found -- skipping training")
            _spec_bar.update(1)
            continue
        _run_training_isolated(spec)
        jax.clear_caches(); gc.collect()
        _spec_bar.update(1)
        _spec_bar.set_postfix(
            arch=spec.arch.name, loss=spec.loss_name,
            solver=_current_info['solver'])
finally:
    _spec_bar.close()
    for _b in list(_step_bars.values()):
        _b.close()
    _step_bars.clear()


In [ ]:
# Per-group loss-curve grids: ARCH_NAMES rows x LOSS_NAMES cols; within
# each panel the 3 solver configs are overlaid. Reads per-spec
# total-loss history from {spec.checkpoint_dir}/losses.npy.
def _plot_group(specs, group_name, phase_label):
    if not specs:
        print(f"[{group_name}] no specs -- skipping plot")
        return
    fig, axes = plt.subplots(
        len(ARCH_NAMES), len(LOSS_NAMES),
        figsize=(4 * len(LOSS_NAMES), 3 * len(ARCH_NAMES)),
        sharex=True, squeeze=False,
    )
    _found_any = False
    for _spec in specs:
        _losses_path = os.path.join(_spec.checkpoint_dir, "losses.npy")
        if not os.path.isfile(_losses_path):
            continue
        _losses = np.load(_losses_path)
        if _losses.size == 0:
            continue
        _found_any = True
        # tail = [..., group, arch, loss, solver]
        _tail = _spec.checkpoint_dir.rstrip("/").split("/")
        _solver = _tail[-1]
        _loss_label = _tail[-2]
        _arch = _tail[-3]
        _ri = ARCH_NAMES.index(_arch) if _arch in ARCH_NAMES else 0
        _ci = LOSS_NAMES.index(_loss_label) if _loss_label in LOSS_NAMES else 0
        axes[_ri][_ci].semilogy(_losses, label=_solver, alpha=0.8)
    # Titles / labels per panel
    for _ri, _arch_name in enumerate(ARCH_NAMES):
        for _ci, _loss_name in enumerate(LOSS_NAMES):
            _ax = axes[_ri][_ci]
            _ax.set_title(f"{_arch_name} / {_loss_name}", fontsize=9)
            _ax.grid(True, which="both", ls=":", alpha=0.4)
            if _ri == len(ARCH_NAMES) - 1:
                _ax.set_xlabel("training step")
            if _ci == 0:
                _ax.set_ylabel("total loss (log)")
            if _ax.lines:
                _ax.legend(fontsize=7, loc="best")
    if not _found_any:
        print(f"[{group_name}] no losses.npy found -- run training first")
    fig.suptitle(f"{group_name} ({phase_label})", fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    os.makedirs(figures_dir, exist_ok=True)
    fig.savefig(
        os.path.join(figures_dir, f"loss_curves_{group_name}.png"),
        dpi=150, bbox_inches="tight",
    )
    plt.show()

_plot_group(_specs_group1, "group1_h2o_short",      "H2O only, short")
_plot_group(_specs_group2, "group2_h2o_c2h2_short", "H2O + C2H2, short")
_plot_group(_specs_group3, "group3_h2o_c2h2_long",  "H2O + C2H2, long")


In [ ]:
# Tidy DataFrame of final loss components per spec. Reads aux_log.pkl and
# train_metadata.json from each spec's checkpoint directory.
_all_specs = list(_specs_group1) + list(_specs_group2) + list(_specs_group3)

def _infer_group(_path):
    _tail = _path.rstrip("/").split("/")
    # tail = [..., group, arch, loss, solver]; group is 4th from the end
    return _tail[-4] if len(_tail) >= 4 else "?"

_rows = []
for _spec in _all_specs:
    _aux_path = os.path.join(_spec.checkpoint_dir, "aux_log.pkl")
    _md_path = os.path.join(_spec.checkpoint_dir, "train_metadata.json")
    if not os.path.isfile(_aux_path):
        continue
    with open(_aux_path, "rb") as _f:
        _aux_log = pickle.load(_f)
    if not _aux_log:
        continue
    _last = _aux_log[-1]
    _aux_dict = _last.get("aux", {}) if isinstance(_last, dict) else {}
    _final_total = float(_last.get("loss", float("nan")))
    if os.path.isfile(_md_path):
        with open(_md_path) as _f:
            _md = json.load(_f)
        _final_total = float(_md.get("final_loss", _final_total))
    _tail = _spec.checkpoint_dir.rstrip("/").split("/")
    _solver = _tail[-1]
    _loss_label = _tail[-2]
    _arch = _tail[-3]
    _rows.append({
        "group": _infer_group(_spec.checkpoint_dir),
        "arch": _arch,
        "loss": _loss_label,
        "solver": _solver,
        "loss_total_final": _final_total,
        "loss_vxc_final":    float(_aux_dict.get("loss_vxc", 0.0)),
        "loss_anchor_final": float(_aux_dict.get("loss_anchor", 0.0)),
    })
_aux_df = pd.DataFrame(_rows)
if len(_aux_df) > 0:
    print(f"Aux inspection: {len(_aux_df)} completed specs")
    print(_aux_df.to_string(index=False))
else:
    print("No training aux logs found yet -- run training first.")


## Section 5 -- Evaluation

Per-spec evaluation sweep. For each of the 72 trained specs (3 groups x 2
archs x 4 losses x 3 solvers), the model checkpoint is re-run on its
training molecules with ``alec.run_test`` to produce per-molecule metrics
and a scalar aggregate. The metrics are:

- ``total_energy`` -- absolute E (Ha) + error-from-CCSD
- ``atomization_energy`` -- AE (kcal/mol) using the Chakravorty atomic
  references; AE reference values come from W4-11 (H2O=232.974,
  C2H2=405.525)
- ``density_rmse`` -- L2 norm of (rho_NN - rho_CCSD)
- ``constraint_violations`` -- count of negativity / Lieb-Oxford violations

Artifacts are written under ``{CHECKPOINT_BASE}/eval/{group}/{arch}/{loss}/{solver}/``
(``aggregate.json`` + ``per_molecule.json`` + ``test_metadata.json``),
mirroring the training directory layout. A per-spec aggregate existing on
disk causes the sweep to skip (unless ``RERUN_EVAL=True``).

After the sweep, the per-molecule JSON outputs are ingested into a single
long-form / tidy ``eval_df`` DataFrame with one row per
``(group, arch, loss, solver, phase_length, molecule, value_name)``
combination; this is the canonical substrate for all Section 6 plots.


In [ ]:
# Per-spec evaluation sweep. Iterates the same concatenated _all_specs list
# as the training loop; per-spec outputs under
# {CHECKPOINT_BASE}/eval/{group}/{arch}/{loss}/{solver}/.
_eval_base = os.path.join(CHECKPOINT_BASE, "eval")
os.makedirs(_eval_base, exist_ok=True)

for _spec in _all_specs:
    _ckpt = os.path.join(_spec.checkpoint_dir, "model.eqx")
    if not os.path.isfile(_ckpt):
        continue
    _tail = _spec.checkpoint_dir.rstrip("/").split("/")
    _solver = _tail[-1]
    _loss_label = _tail[-2]
    _arch = _tail[-3]
    _group = (
        "group1" if _spec in _specs_group1
        else "group2" if _spec in _specs_group2
        else "group3"
    )
    _out = os.path.join(_eval_base, _group, _arch, _loss_label, _solver)
    if not RERUN_EVAL and os.path.isfile(os.path.join(_out, "aggregate.json")):
        continue
    _ae_ref = {"H2O": H2O_AE_REF_KCALMOL}
    if _group != "group1":
        _ae_ref["C2H2"] = C2H2_AE_REF_KCALMOL
    _test_spec = alec.TestSpec.from_dicts(
        arch=alec.get_architecture(_arch),
        model_checkpoint=_ckpt,
        molecules=tuple(_spec.molecules),
        metrics=("total_energy", "atomization_energy", "density_rmse", "constraint_violations"),
        metric_kwargs={"atomization_energy": {"reference_ae_kcalmol": _ae_ref}},
        atom_energies=ATOMIC_ENERGIES_CHAKRAVORTY,
        output_dir=_out,
        solver_config=SOLVER_CONFIGS[_solver],
        pbe_anchor_weight=_spec.pbe_anchor_weight,
        pbe_anchor_sample=_spec.pbe_anchor_sample,
    )
    alec.run_test(_test_spec)
    # Release compiled XLA artifacts between eval runs to prevent LLVM OOM.
    jax.clear_caches(); gc.collect()

_n_done = sum(
    1 for _s in _all_specs
    if os.path.isfile(os.path.join(
        _eval_base,
        "group1" if _s in _specs_group1 else "group2" if _s in _specs_group2 else "group3",
        _s.checkpoint_dir.rstrip("/").split("/")[-3],
        _s.checkpoint_dir.rstrip("/").split("/")[-2],
        _s.checkpoint_dir.rstrip("/").split("/")[-1],
        "aggregate.json",
    ))
)
print(f"Evaluation complete: {_n_done} / {len(_all_specs)} aggregates on disk")


In [ ]:
# Build eval_df: one row per (group, arch, loss, solver, phase_length,
# molecule, value_name). Numeric scalars from per_molecule.json become
# rows; non-numeric fields (molecule/name) are skipped.
_rows = []
_parq = os.path.join(CHECKPOINT_BASE, "eval_df.parquet")
if not RERUN_EVAL and os.path.isfile(_parq):
    eval_df = pd.read_parquet(_parq)
    print(f"Using cached {_parq} ({len(eval_df)} rows)")
else:
    for _spec in _all_specs:
        _tail = _spec.checkpoint_dir.rstrip("/").split("/")
        _solver = _tail[-1]
        _loss_label = _tail[-2]
        _arch = _tail[-3]
        _group = (
            "group1" if _spec in _specs_group1
            else "group2" if _spec in _specs_group2
            else "group3"
        )
        _phase = "short" if _spec.n_steps == TRAIN_N_STEPS_SHORT else "long"
        _out = os.path.join(CHECKPOINT_BASE, "eval", _group, _arch, _loss_label, _solver)
        _pm_path = os.path.join(_out, "per_molecule.json")
        if not os.path.isfile(_pm_path):
            continue
        with open(_pm_path) as _f:
            _pm = json.load(_f)
        for _row in _pm:
            _mol = _row.get("name") or _row.get("molecule")
            for _k, _v in _row.items():
                if _k in ("name", "molecule"):
                    continue
                if isinstance(_v, bool):
                    # bool is a subtype of int; skip so boolean flags don't
                    # leak into the numeric value column.
                    continue
                if isinstance(_v, (int, float)):
                    _rows.append({
                        "group":        _group,
                        "arch":         _arch,
                        "loss":         _loss_label,
                        "solver":       _solver,
                        "phase_length": _phase,
                        "molecule":     _mol,
                        "value_name":   _k,
                        "value":        float(_v),
                    })
    eval_df = pd.DataFrame(_rows)
    eval_df.to_parquet(_parq)
    print(f"Wrote {_parq} ({len(eval_df)} rows)")


In [ ]:
# V_xc efficacy: L1 vs L3 on Group 2 short. Answers "is V_xc doing anything?"
_g2 = eval_df[eval_df.group == "group2"]
_g2_l1 = _g2[_g2.loss == "L1_B"]
_g2_l3 = _g2[_g2.loss == "L3_balanced_vxc"]

fig, axes = plt.subplots(len(ARCH_NAMES), 3, figsize=(12, 4 * len(ARCH_NAMES)),
                         squeeze=False)
for _ri, _arch in enumerate(ARCH_NAMES):
    for _ci, _metric in enumerate(["abs_ae_error", "density_rmse", "loss_vxc"]):
        _d1 = _g2_l1[(_g2_l1.arch == _arch) & (_g2_l1.value_name == _metric)]
        _d3 = _g2_l3[(_g2_l3.arch == _arch) & (_g2_l3.value_name == _metric)]
        _x = np.arange(len(SOLVER_LABELS)); _w = 0.4
        _vals1 = (_d1.groupby("solver")["value"].mean()
                    .reindex(SOLVER_LABELS).fillna(0.0).values)
        _vals3 = (_d3.groupby("solver")["value"].mean()
                    .reindex(SOLVER_LABELS).fillna(0.0).values)
        axes[_ri][_ci].bar(_x - _w/2, _vals1, width=_w, label="L1 (no V_xc)")
        axes[_ri][_ci].bar(_x + _w/2, _vals3, width=_w, label="L3 (V_xc)")
        axes[_ri][_ci].set_xticks(_x)
        axes[_ri][_ci].set_xticklabels(SOLVER_LABELS, rotation=30, fontsize=7)
        axes[_ri][_ci].set_title(f"{_arch} | {_metric}", fontsize=9)
        if _ci == 0: axes[_ri][_ci].legend(fontsize=7)
fig.suptitle("V_xc efficacy (L1 vs L3, Group 2 short)")
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "vxc_efficacy.png"), dpi=120); plt.show()


In [ ]:
# Anchor-effect: L3 -> L4 ΔAE across (arch, solver, group).
fig, axes = plt.subplots(len(ARCH_NAMES), 3, figsize=(14, 4 * len(ARCH_NAMES)),
                         squeeze=False)
for _ri, _arch in enumerate(ARCH_NAMES):
    for _ci, _grp in enumerate(["group1", "group2", "group3"]):
        _slice = eval_df[
            (eval_df.group == _grp) & (eval_df.arch == _arch)
            & (eval_df.value_name == "abs_ae_error")
        ]
        _deltas = []; _labels = []
        for _s in SOLVER_LABELS:
            _off = _slice[(_slice.loss == "L3_balanced_vxc")
                          & (_slice.solver == _s)]["value"].mean()
            _on = _slice[(_slice.loss == "L4_balanced_vxc_anchor")
                         & (_slice.solver == _s)]["value"].mean()
            _deltas.append((_off if pd.notna(_off) else 0.0)
                           - (_on if pd.notna(_on) else 0.0))
            _labels.append(_s)
        _x = np.arange(len(_deltas))
        axes[_ri][_ci].bar(_x, _deltas,
                           color=["green" if _d > 0 else "red" for _d in _deltas])
        axes[_ri][_ci].axhline(0, color="k", lw=0.5)
        axes[_ri][_ci].set_xticks(_x); axes[_ri][_ci].set_xticklabels(_labels, fontsize=7)
        axes[_ri][_ci].set_title(f"{_arch} | {_grp}", fontsize=9)
        axes[_ri][_ci].set_ylabel("ΔAE (L3 − L4; + = anchor helps)")
fig.suptitle("Anchor effect: AE(L3 V_xc) − AE(L4 V_xc+anchor)")
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "anchor_effect.png"), dpi=120); plt.show()


## Section 6 -- Transfer-learning Evaluation

Evaluates every trained spec on W4-11 molecules held out from training.
Primary set is small and chemically close (H2 / OH / CH4); secondary set
spans a broader chemistry (NH3 / HF / CO2 / NH2). NH2 is a UKS doublet.

Geometries + AE references: W4-11. No OEP on transfer molecules -- V_xc
matching is a training-only regularizer.


In [ ]:
# Primary transfer set: {H2, OH, CH4} on W4-11 geometries.
# OH is UKS (doublet); H2 and CH4 are RKS closed shell.
TRANSFER_PRIMARY = (
    {
        "name": "H2",
        "atom": "H  0.000000  0.000000  0.370946; H  0.000000  0.000000 -0.370946",
        "spin": 0,
        "ae_ref_kcalmol": 109.493,
        "comp": (("H", 2),),
    },
    {
        "name": "OH",
        "atom": "O  0.000000  0.000000  0.107851; H  0.000000  0.000000 -0.862809",
        "spin": 1,
        "ae_ref_kcalmol": 107.208,
        "comp": (("O", 1), ("H", 1)),
    },
    {
        "name": "CH4",
        "atom": ("H 0.628099 0.628099 0.628099; C 0 0 0; "
                 "H -0.628099 -0.628099 0.628099; "
                 "H -0.628099 0.628099 -0.628099; "
                 "H 0.628099 -0.628099 -0.628099"),
        "spin": 0,
        "ae_ref_kcalmol": 420.420,
        "comp": (("C", 1), ("H", 4)),
    },
)


def _gen_transfer_npz(m, out_dir):
    # Generate {name}.npz + {name}_metadata.json for a transfer molecule.
    # Runs PBE + HF + CCSD on the W4-11 geometry, extracts AO-basis CCSD DM
    # (spin-resolved for UKS), computes rho_ccsd on the PBE grid. No OEP.
    # Guard: skip if both artifacts already exist.
    _npz = os.path.join(out_dir, f"{m['name']}.npz")
    _meta = os.path.join(out_dir, f"{m['name']}_metadata.json")
    if os.path.isfile(_npz) and os.path.isfile(_meta):
        print(f"Using cached {_npz}")
        return
    _mol = gto.M(atom=m["atom"], basis=BASIS, charge=0, spin=m["spin"], verbose=0)
    if m["spin"]:
        _mf_pbe = dft.UKS(_mol); _mf_pbe.xc = "pbe"; _mf_pbe.grids.level = GRID_LEVEL
        _mf_pbe.kernel(); E_pbe = float(_mf_pbe.e_tot)
        _mf_hf = scf.UHF(_mol); _mf_hf.kernel(); E_hf = float(_mf_hf.e_tot)
        _cc = cc.UCCSD(_mf_hf); _cc.kernel()
        E_ccsd = float(_mf_hf.e_tot + _cc.e_corr)
        dm_mo_ab = _cc.make_rdm1()
        Ca, Cb = _mf_hf.mo_coeff
        dm_ao_a = Ca @ dm_mo_ab[0] @ Ca.T
        dm_ao_b = Cb @ dm_mo_ab[1] @ Cb.T
        dm_ao = np.stack([dm_ao_a, dm_ao_b], axis=0)   # (2, nao, nao)
        _ao = _mf_pbe._numint.eval_ao(_mol, _mf_pbe.grids.coords, deriv=0)
        rho_ccsd = np.einsum("ij,gi,gj->g", dm_ao_a + dm_ao_b, _ao, _ao)
    else:
        _mf_pbe = dft.RKS(_mol); _mf_pbe.xc = "pbe"; _mf_pbe.grids.level = GRID_LEVEL
        _mf_pbe.kernel(); E_pbe = float(_mf_pbe.e_tot)
        _mf_hf = scf.RHF(_mol); _mf_hf.kernel(); E_hf = float(_mf_hf.e_tot)
        _cc = cc.CCSD(_mf_hf); _cc.kernel()
        E_ccsd = float(_mf_hf.e_tot + _cc.e_corr)
        dm_mo = _cc.make_rdm1()
        C = _mf_hf.mo_coeff
        dm_ao = C @ dm_mo @ C.T
        _ao = _mf_pbe._numint.eval_ao(_mol, _mf_pbe.grids.coords, deriv=0)
        rho_ccsd = np.einsum("ij,gi,gj->g", dm_ao, _ao, _ao)
    np.savez(_npz,
             dm_target=dm_ao,
             rho_ref_grid=rho_ccsd,
             ref_density_method="ccsd",
             E_ref_literature=E_ccsd)
    with open(_meta, "w") as _f:
        json.dump({"E_hf_total": E_hf, "E_ccsd_total": E_ccsd,
                   "E_pbe_total": E_pbe,
                   "ae_ref_kcalmol": m["ae_ref_kcalmol"]}, _f, indent=2)
    print(f"Wrote {_npz}  E_ccsd={E_ccsd:+.4f}  AE_ref={m['ae_ref_kcalmol']:.3f} kcal/mol")


for _m in TRANSFER_PRIMARY:
    _gen_transfer_npz(_m, transfer_primary)


In [ ]:
# Secondary transfer set: {NH3, HF, CO2, NH2} on W4-11 geometries.
# NH2 is a UKS doublet radical; the rest are closed-shell RKS.
TRANSFER_SECONDARY = (
    {
        "name": "NH3",
        "atom": ("N 0 0 0.116671; H 0 0.934724 -0.272232; "
                 "H 0.809495 -0.467362 -0.272232; "
                 "H -0.809495 -0.467362 -0.272232"),
        "spin": 0,
        "ae_ref_kcalmol": 298.018,
        "comp": (("N", 1), ("H", 3)),
    },
    {
        "name": "HF",
        "atom": "F 0 0 0.091577; H 0 0 -0.824192",
        "spin": 0,
        "ae_ref_kcalmol": 141.640,
        "comp": (("H", 1), ("F", 1)),
    },
    {
        "name": "CO2",
        "atom": "C 0 0 0; O 0 0 1.162600; O 0 0 -1.162600",
        "spin": 0,
        "ae_ref_kcalmol": 390.141,
        "comp": (("C", 1), ("O", 2)),
    },
    {
        "name": "NH2",
        "atom": ("N 0 0 0.142235; H 0 0.800646 -0.497821; "
                 "H 0 -0.800646 -0.497821"),
        "spin": 1,
        "ae_ref_kcalmol": 182.591,
        "comp": (("N", 1), ("H", 2)),
    },
)

for _m in TRANSFER_SECONDARY:
    _gen_transfer_npz(_m, transfer_secondary)


In [ ]:
# Primary transfer test loop. Reads trained checkpoint for each of the 72
# specs, runs alec.run_test on each transfer molecule, aggregates to tidy
# DataFrame. pbe_anchor_weight=0 / pbe_anchor_sample=None for transfer
# (anchor is a training regularizer only).
def _run_transfer_eval(mols_list, out_dir, parquet_name):
    _parq = os.path.join(CHECKPOINT_BASE, parquet_name)
    if not RERUN_EVAL and os.path.isfile(_parq):
        _df = pd.read_parquet(_parq)
        print(f"Using cached {_parq} ({len(_df)} rows)")
        return _df
    _ae_ref = {_m["name"]: _m["ae_ref_kcalmol"] for _m in mols_list}
    _mol_specs = tuple(
        alec.MoleculeSpec(
            name=_m["name"], atom=_m["atom"], basis=BASIS,
            charge=0, spin=_m["spin"], grid_level=GRID_LEVEL,
            atom_composition=_m["comp"],
            external_data_path=os.path.join(out_dir, f"{_m['name']}.npz"),
        )
        for _m in mols_list
    )
    _rows = []
    for _spec in _all_specs:
        _ckpt = os.path.join(_spec.checkpoint_dir, "model.eqx")
        if not os.path.isfile(_ckpt):
            continue
        _tail = _spec.checkpoint_dir.rstrip("/").split("/")
        _solver = _tail[-1]
        _loss_label = _tail[-2]
        _arch = _tail[-3]
        _group = (
            "group1" if _spec in _specs_group1
            else "group2" if _spec in _specs_group2
            else "group3"
        )
        _out = os.path.join(CHECKPOINT_BASE, "transfer_eval",
                            parquet_name.replace(".parquet", ""),
                            _group, _arch, _loss_label, _solver)
        _agg_path = os.path.join(_out, "aggregate.json")
        _pm_path = os.path.join(_out, "per_molecule.json")
        if RERUN_EVAL or not os.path.isfile(_agg_path):
            _test_spec = alec.TestSpec.from_dicts(
                arch=alec.get_architecture(_arch),
                model_checkpoint=_ckpt,
                molecules=_mol_specs,
                metrics=("total_energy", "atomization_energy", "density_rmse"),
                metric_kwargs={"atomization_energy":
                               {"reference_ae_kcalmol": _ae_ref}},
                atom_energies=ATOMIC_ENERGIES_CHAKRAVORTY,
                output_dir=_out,
                solver_config=SOLVER_CONFIGS[_solver],
                pbe_anchor_weight=0.0,
                pbe_anchor_sample=None,
            )
            alec.run_test(_test_spec)
            jax.clear_caches(); gc.collect()
        if not os.path.isfile(_pm_path):
            continue
        with open(_pm_path) as _f:
            _pm = json.load(_f)
        for _row in _pm:
            _mol = _row.get("name") or _row.get("molecule")
            for _k, _v in _row.items():
                if _k in ("name", "molecule"):
                    continue
                if isinstance(_v, bool):
                    continue
                if isinstance(_v, (int, float)):
                    _rows.append({
                        "group":      _group,
                        "arch":       _arch,
                        "loss":       _loss_label,
                        "solver":     _solver,
                        "molecule":   _mol,
                        "value_name": _k,
                        "value":      float(_v),
                    })
    _df = pd.DataFrame(_rows)
    _df.to_parquet(_parq)
    print(f"Wrote {_parq} ({len(_df)} rows)")
    return _df


transfer_primary_df = _run_transfer_eval(
    TRANSFER_PRIMARY, transfer_primary, "transfer_primary_df.parquet",
)
print(f"transfer_primary_df: {len(transfer_primary_df)} rows")


In [ ]:
# Secondary transfer test loop. Same shape as primary; reuses the helper
# _run_transfer_eval (which calls alec.run_test under the hood) defined
# in the previous cell.
transfer_secondary_df = _run_transfer_eval(
    TRANSFER_SECONDARY, transfer_secondary, "transfer_secondary_df.parquet",
)
print(f"transfer_secondary_df: {len(transfer_secondary_df)} rows")


In [ ]:
# Cross-mol MAE (kcal/mol) bar chart on primary transfer set. Aggregated
# over molecules, grouped by (group, arch); 4 loss bars each, log-y.
fig, axes = plt.subplots(
    len(ARCH_NAMES), 3, figsize=(14, 4 * len(ARCH_NAMES)), squeeze=False,
)
_x = np.arange(len(LOSS_NAMES))
_w = 0.22
for _ri, _arch in enumerate(ARCH_NAMES):
    for _ci, _grp in enumerate(["group1", "group2", "group3"]):
        _ax = axes[_ri][_ci]
        _slice = transfer_primary_df[
            (transfer_primary_df.group == _grp)
            & (transfer_primary_df.arch == _arch)
            & (transfer_primary_df.value_name == "abs_ae_error")
        ]
        for _si, _solver in enumerate(SOLVER_LABELS):
            _vals = []
            for _loss in LOSS_NAMES:
                _d = _slice[(_slice.solver == _solver) & (_slice.loss == _loss)]
                _mae = _d["value"].mean() if len(_d) else 0.0
                _vals.append(_mae if pd.notna(_mae) else 0.0)
            _ax.bar(_x + (_si - 1) * _w, _vals, width=_w, label=_solver)
        _ax.set_xticks(_x)
        _ax.set_xticklabels(LOSS_NAMES, rotation=25, fontsize=7)
        _ax.set_yscale("log")
        _ax.set_title(f"{_arch} | {_grp}", fontsize=9)
        _ax.grid(True, which="both", ls=":", alpha=0.4)
        if _ci == 0:
            _ax.set_ylabel("MAE abs_ae_error (kcal/mol, log)")
            _ax.legend(fontsize=7, title="solver")
fig.suptitle("Primary transfer: cross-mol MAE on {H2, OH, CH4}")
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(os.path.join(figures_dir, "transfer_primary_mae.png"),
            dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Cross-mol MAE (kcal/mol) bar chart on secondary transfer set. Aggregated
# over molecules, grouped by (group, arch); 4 loss bars each, log-y.
fig, axes = plt.subplots(
    len(ARCH_NAMES), 3, figsize=(14, 4 * len(ARCH_NAMES)), squeeze=False,
)
_x = np.arange(len(LOSS_NAMES))
_w = 0.22
for _ri, _arch in enumerate(ARCH_NAMES):
    for _ci, _grp in enumerate(["group1", "group2", "group3"]):
        _ax = axes[_ri][_ci]
        _slice = transfer_secondary_df[
            (transfer_secondary_df.group == _grp)
            & (transfer_secondary_df.arch == _arch)
            & (transfer_secondary_df.value_name == "abs_ae_error")
        ]
        for _si, _solver in enumerate(SOLVER_LABELS):
            _vals = []
            for _loss in LOSS_NAMES:
                _d = _slice[(_slice.solver == _solver) & (_slice.loss == _loss)]
                _mae = _d["value"].mean() if len(_d) else 0.0
                _vals.append(_mae if pd.notna(_mae) else 0.0)
            _ax.bar(_x + (_si - 1) * _w, _vals, width=_w, label=_solver)
        _ax.set_xticks(_x)
        _ax.set_xticklabels(LOSS_NAMES, rotation=25, fontsize=7)
        _ax.set_yscale("log")
        _ax.set_title(f"{_arch} | {_grp}", fontsize=9)
        _ax.grid(True, which="both", ls=":", alpha=0.4)
        if _ci == 0:
            _ax.set_ylabel("MAE abs_ae_error (kcal/mol, log)")
            _ax.legend(fontsize=7, title="solver")
fig.suptitle("Secondary transfer: cross-mol MAE on {NH3, HF, CO2, NH2}")
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(os.path.join(figures_dir, "transfer_secondary_mae.png"),
            dpi=120, bbox_inches="tight")
plt.show()


## Section 7 -- F_x(s) Drift Diagnostic (headline)

Step 5 finding: the trained NN F_x(s) drifts away from PBE at s > 0.7 on
CH4 grid points, and this drift correlates with the CH4 transfer gap.
Step 6's two candidate fixes (add C2H2 to training; PBE-anchor
regularization on synthetic (rho, s) samples) are evaluated by sampling
F_x(s) on three molecular grids spanning three regimes:

* **Panel B / CH4**  -- transfer-reference molecule (the step-5 finding
  that motivates step 6).
* **Panel B / C2H2** -- in-training molecule for groups 2 + 3; answers
  "does C2H2 in the batch stabilize F_x where the batch samples it?"
* **Panel C / C2H4** -- held-out generalization probe; not in any
  training or transfer set, tests whether (a) data expansion or (b)
  anchor regularization closes the drift on a genuinely unseen molecule.

For each panel the analytic PBE curve (solid black) is the reference;
pretrained baselines (green) show where fine-tuning starts; fine-tuned
models (blue / orange / red for groups 1 / 2 / 3) show where fine-tuning
ends.


In [ ]:
# F_x(s) drift Panel B: CH4 (transfer ref) + C2H2 (in-training). Samples
# F_x on each molecule's PBE grid for all 72 trained models + per-arch
# pretrained baselines + the analytic PBE reference.
from xcquinox.alec.oneshot import _nn_fx_local_uks
from xcquinox.alec.models import AlecGGAModel
from xcquinox.alec.networks import create_network_pair


def _load_full_model_from_ckpt(_ckpt_path, _arch_name):
    # Canonical pattern (matches run_test @ evaluation.py:215-218).
    _arch_cfg = alec.get_architecture(_arch_name)
    _skel = AlecGGAModel.from_arch(_arch_cfg, seed=0)
    return eqx.tree_deserialise_leaves(_ckpt_path, _skel)


def _load_pretrain_model(_pretrain_dir, _arch_name):
    # Pretrain saves xnet.eqx + cnet.eqx separately (step-3 pretrain.py);
    # combine into a full AlecGGAModel for F_x sampling.
    _arch_cfg = alec.get_architecture(_arch_name)
    _xskel, _cskel = create_network_pair(_arch_cfg, seed=0)
    _xnet = eqx.tree_deserialise_leaves(
        os.path.join(_pretrain_dir, _arch_name, "xnet.eqx"), _xskel,
    )
    _cnet = eqx.tree_deserialise_leaves(
        os.path.join(_pretrain_dir, _arch_name, "cnet.eqx"), _cskel,
    )
    return AlecGGAModel.from_arch(_arch_cfg, xnet=_xnet, cnet=_cnet)


def _sample_fx_on_molecule(_model, _atom_str, _spin):
    # Runs PBE on the molecule, extracts density + its gradient on the PBE
    # grid, computes reduced gradient s, then evaluates the NN F_x via the
    # spin-scaled UKS helper (matches SCF-time convention).
    _mol = gto.M(atom=_atom_str, basis=BASIS, charge=0, spin=_spin, verbose=0)
    _mf = dft.UKS(_mol) if _spin else dft.RKS(_mol)
    _mf.xc = "pbe"
    _mf.grids.level = GRID_LEVEL
    _mf.kernel()
    _dm = _mf.make_rdm1()
    if _spin:
        _dm = _dm[0] + _dm[1]
    _coords = _mf.grids.coords
    _ao_deriv = _mf._numint.eval_ao(_mol, _coords, deriv=1)
    _ao = _ao_deriv[0]
    _ao_xyz = _ao_deriv[1:4]
    _rho = jnp.einsum("ij,gi,gj->g", _dm, _ao, _ao)
    # grad_rho_k = 2 * sum_{ij} D_ij * (d_k phi_i) * phi_j  (hermitian D).
    _grad_rho = 2.0 * jnp.einsum("ij,dgi,gj->gd", _dm, _ao_xyz, _ao)
    _grad_mag = jnp.linalg.norm(_grad_rho, axis=1)
    _kF = (3.0 * jnp.pi ** 2) ** (1.0 / 3.0)
    _s = _grad_mag / (2.0 * _kF * jnp.clip(_rho, 1e-12, None) ** (4.0 / 3.0))
    _fx_nn = _nn_fx_local_uks(_model, _rho / 2.0, _rho / 2.0, _s)
    return np.asarray(_s), np.asarray(_fx_nn)


def _fx_pbe_analytic(_s):
    # PBE: F_x(s) = 1 + kappa - kappa/(1 + mu*s^2/kappa), with
    # kappa=0.804 (Lieb-Oxford bound) and mu=0.21951 (Perdew et al. 1996).
    _kappa = 0.804
    _mu = 0.21951
    return 1.0 + _kappa - _kappa / (1.0 + _mu * _s ** 2 / _kappa)


_PROBE_MOLS = [
    ("CH4",
     "C 0 0 0; H 0.628099 0.628099 0.628099; H -0.628099 -0.628099 0.628099; "
     "H -0.628099 0.628099 -0.628099; H 0.628099 -0.628099 -0.628099",
     0),
    ("C2H2", C2H2_ATOM, 0),
]

_GROUP_COLORS = {"group1": "tab:blue", "group2": "tab:orange", "group3": "tab:red"}
_s_ref = np.linspace(0.01, 15.0, 200)
_fx_ref = _fx_pbe_analytic(_s_ref)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for _ax, (_nm, _atom, _spin) in zip(axes, _PROBE_MOLS):
    # Analytic PBE reference.
    _ax.plot(_s_ref, _fx_ref, "k-", lw=2, label="PBE (analytic)")

    # Per-arch pretrained baselines.
    for _arch in ARCH_NAMES:
        _pre_ckdir = os.path.join(pretrain_dir, _arch)
        if not (os.path.isfile(os.path.join(_pre_ckdir, "xnet.eqx"))
                and os.path.isfile(os.path.join(_pre_ckdir, "cnet.eqx"))):
            continue
        try:
            _m = _load_pretrain_model(pretrain_dir, _arch)
            _sv, _fx = _sample_fx_on_molecule(_m, _atom, _spin)
        except Exception as _e:
            print(f"[Panel B {_nm}] pretrain {_arch} skipped: {_e}")
            continue
        _ax.scatter(_sv, _fx, s=2, alpha=0.3, color="green",
                    label="pretrained" if _arch == ARCH_NAMES[0] else None)
        jax.clear_caches(); gc.collect()

    # Fine-tuned models.
    _seen_groups = set()
    for _spec in _all_specs:
        _ckpt = os.path.join(_spec.checkpoint_dir, "model.eqx")
        if not os.path.isfile(_ckpt):
            continue
        _group = (
            "group1" if _spec in _specs_group1
            else "group2" if _spec in _specs_group2
            else "group3"
        )
        try:
            _m = _load_full_model_from_ckpt(_ckpt, _spec.arch.name)
            _sv, _fx = _sample_fx_on_molecule(_m, _atom, _spin)
        except Exception as _e:
            print(f"[Panel B {_nm}] {_spec.checkpoint_dir} skipped: {_e}")
            continue
        _lbl = _group if _group not in _seen_groups else None
        _seen_groups.add(_group)
        _ax.scatter(_sv, _fx, s=2, alpha=0.15,
                    color=_GROUP_COLORS[_group], label=_lbl)
        jax.clear_caches(); gc.collect()

    _ax.set_xscale("log")
    _ax.set_xlim(0.01, 15)
    _ax.set_xlabel("reduced gradient s (log)")
    _ax.set_title(f"F_x(s) sampled at {_nm} grid")
    _ax.grid(True, which="both", ls=":", alpha=0.4)

axes[0].set_ylabel(r"exchange enhancement $F_x(s)$")
axes[0].legend(fontsize=8, loc="upper left", framealpha=0.9)
fig.suptitle("Panel B: F_x(s) drift at CH4 (transfer ref) + C2H2 (in-training)",
             fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(os.path.join(figures_dir, "fx_drift_panel_B.png"),
            dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# F_x(s) drift Panel C: C2H4 (ethylene) held-out generalization probe.
# Not in any training or transfer set -- tests whether the trained-model
# F_x deformation persists on a truly unseen molecule. Reuses helpers
# (_sample_fx_on_molecule, _fx_pbe_analytic, _load_pretrain_model,
# _load_full_model_from_ckpt) defined in the previous cell.
C2H4_ATOM = (
    "C  0.000000  0.000000   0.667100; "
    "C  0.000000  0.000000  -0.667100; "
    "H  0.000000  0.923404  -1.231634; "
    "H  0.000000 -0.923404  -1.231634; "
    "H  0.000000  0.923404   1.231634; "
    "H  0.000000 -0.923404   1.231634"
)

_s_ref_c = np.linspace(0.01, 15.0, 200)
_fx_ref_c = _fx_pbe_analytic(_s_ref_c)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(_s_ref_c, _fx_ref_c, "k-", lw=2, label="PBE (analytic)")

# Per-arch pretrained baselines.
for _arch in ARCH_NAMES:
    _pre_ckdir = os.path.join(pretrain_dir, _arch)
    if not (os.path.isfile(os.path.join(_pre_ckdir, "xnet.eqx"))
            and os.path.isfile(os.path.join(_pre_ckdir, "cnet.eqx"))):
        continue
    try:
        _m = _load_pretrain_model(pretrain_dir, _arch)
        _sv, _fx = _sample_fx_on_molecule(_m, C2H4_ATOM, 0)
    except Exception as _e:
        print(f"[Panel C] pretrain {_arch} skipped: {_e}")
        continue
    ax.scatter(_sv, _fx, s=2, alpha=0.3, color="green",
               label="pretrained" if _arch == ARCH_NAMES[0] else None)
    jax.clear_caches(); gc.collect()

# Fine-tuned models.
_seen_groups_c = set()
for _spec in _all_specs:
    _ckpt = os.path.join(_spec.checkpoint_dir, "model.eqx")
    if not os.path.isfile(_ckpt):
        continue
    _group = (
        "group1" if _spec in _specs_group1
        else "group2" if _spec in _specs_group2
        else "group3"
    )
    try:
        _m = _load_full_model_from_ckpt(_ckpt, _spec.arch.name)
        _sv, _fx = _sample_fx_on_molecule(_m, C2H4_ATOM, 0)
    except Exception as _e:
        print(f"[Panel C] {_spec.checkpoint_dir} skipped: {_e}")
        continue
    _lbl = _group if _group not in _seen_groups_c else None
    _seen_groups_c.add(_group)
    ax.scatter(_sv, _fx, s=2, alpha=0.15,
               color=_GROUP_COLORS[_group], label=_lbl)
    jax.clear_caches(); gc.collect()

ax.set_xscale("log")
ax.set_xlim(0.01, 15)
ax.set_xlabel("reduced gradient s (log)")
ax.set_ylabel(r"exchange enhancement $F_x(s)$")
ax.set_title("Panel C: F_x(s) at C2H4 grid (held-out generalization probe)")
ax.grid(True, which="both", ls=":", alpha=0.4)
ax.legend(fontsize=8, loc="upper left", framealpha=0.9)
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "fx_drift_panel_C.png"),
            dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# SCF convergence aggregate. Reads each spec's aux_log.pkl and tries to
# extract a per-step cycles_run field (canonical SCF convergence metric;
# see SCFResult @ solver.py:147-152). The default xcquinox training loop
# does NOT record cycles_run in aux (the aux dict is the loss components
# returned by compute_components), and train_metadata.json is scalar-only
# -- so on the default build this cell produces a "no convergence data"
# placeholder. Once a future training loop records cycles_run per step
# under aux_log[*]['aux']['cycles_run'], the bar chart populates.
_conv_rows = []
for _spec in _all_specs:
    _ckpt = os.path.join(_spec.checkpoint_dir, "model.eqx")
    if not os.path.isfile(_ckpt):
        continue
    _md_path = os.path.join(_spec.checkpoint_dir, "train_metadata.json")
    _aux_path = os.path.join(_spec.checkpoint_dir, "aux" + "_log" + ".pkl")
    # Derive the solver label from the checkpoint_dir trailing segment
    # (matches the layout in cells 18-20).
    _solver_label = os.path.basename(_spec.checkpoint_dir.rstrip("/"))
    _cycles_vals = []
    # First try train_metadata.json -- scalar-only today but cheap to probe.
    if os.path.isfile(_md_path):
        try:
            with open(_md_path) as _f:
                _md = json.load(_f)
            if "cycles_run" in _md and _md["cycles_run"] is not None:
                _cycles_vals.append(float(_md["cycles_run"]))
        except Exception:
            pass
    # Then try aux_log.pkl -- the canonical per-step record.
    if not _cycles_vals and os.path.isfile(_aux_path):
        try:
            with open(_aux_path, "rb") as _f:
                _log = pickle.load(_f)
            for _entry in _log:
                _aux = _entry.get("aux") if isinstance(_entry, dict) else None
                if isinstance(_aux, dict) and "cycles_run" in _aux:
                    _cv = _aux["cycles_run"]
                    try:
                        _cycles_vals.append(float(_cv))
                    except (TypeError, ValueError):
                        continue
        except Exception:
            pass
    if _cycles_vals:
        _conv_rows.append({
            "solver": _solver_label,
            "cycles_run": float(np.mean(_cycles_vals)),
        })

_conv_df = pd.DataFrame(_conv_rows)
fig, ax = plt.subplots(figsize=(7, 4))
if len(_conv_df) == 0:
    # No convergence data captured on this build. Emit a placeholder so
    # the notebook does not hard-fail when aux_log doesn't record
    # cycles_run (the default today).
    ax.text(0.5, 0.5,
            "no convergence data\\n"
            "(cycles_run not recorded in aux_log / train_metadata on this build)",
            ha="center", va="center", fontsize=11,
            transform=ax.transAxes, color="gray")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title("SCF convergence across training specs")
    print("[Cell 39] no cycles_run recorded -- placeholder plot")
else:
    _grp = _conv_df.groupby("solver")["cycles_run"]
    _mean = _grp.mean()
    _std = _grp.std().fillna(0.0)
    # Preserve SOLVER_LABELS order when possible, else fall back to
    # whatever solver labels showed up.
    _ordered = [_s for _s in SOLVER_LABELS if _s in _mean.index]
    _extras = [_s for _s in _mean.index if _s not in SOLVER_LABELS]
    _ordered.extend(_extras)
    _mean = _mean.reindex(_ordered)
    _std = _std.reindex(_ordered)
    _x = np.arange(len(_mean))
    ax.bar(_x, _mean.values, yerr=_std.values, capsize=6,
           color="tab:steelblue", edgecolor="k", linewidth=0.5)
    ax.set_xticks(_x)
    ax.set_xticklabels(list(_mean.index), rotation=20, fontsize=9)
    ax.set_ylabel("cycles_run (mean +/- stddev)")
    ax.set_title("SCF convergence across training specs")
    ax.grid(True, axis="y", ls=":", alpha=0.4)
    print(f"[Cell 39] plotted SCF convergence over {len(_conv_df)} specs")

fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "scf_convergence.png"),
            dpi=120, bbox_inches="tight")
plt.show()


## Section 7 -- Findings

Populate post-run. Template:
- H1 (data fix): ...
- H2 (regularization fix): ...
- H3 (interaction): ...
- H4 (overfitting): ...
- H5 (V_xc necessity): ...


## Section 8 -- Step 7 Roadmap

Skeleton -- body depends on step-6 results. Candidate directions:
1. If data fix works: widen training to W4-11 subset.
2. If PBE-anchor works: sweep w_anchor in {1e-4, 1e-3, 1e-2}.
3. If overfitting confirmed: test early-stopping criteria.


## Closing

End of step-6 notebook. Regenerate from
`notebooks/_build_step6_notebook.py` -- never hand-edit.
